# Dev Playground

Prototypes `flowcyto` analysis code (FCS parsing, compensation, gating,
transforms, clustering, viz) against real files in `data/` before it moves
into `src/flowcyto/` and the Streamlit app.

Free and open-source only (FlowKit, pandas, Plotly) — no paid APIs.

## Setup (run once)

Installs `requirements.txt` plus notebook-only extras (`ipykernel`,
`ipywidgets` — for the interactive section below). Safe to re-run.

In [1]:
%pip install -q -r ../requirements.txt ipykernel ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import flowkit as fk
import pandas as pd
import plotly.express as px

DATA_DIR = Path("../data")

## Pick a sample file

Lists the `.fcs` files in `data/`.

In [3]:
fcs_files = sorted(DATA_DIR.glob("*.fcs"))
fcs_files

[PosixPath('../data/Comp_CD11c APC_D3_D03_020.fcs'),
 PosixPath('../data/Comp_CD123 BB660_F2_F02_014.fcs'),
 PosixPath('../data/Comp_CD14 BV510_G1_G01_007.fcs'),
 PosixPath('../data/Comp_CD19 PE-Cy5_A3_A03_017.fcs'),
 PosixPath('../data/Comp_CD20 BUV805_E1_E01_005.fcs'),
 PosixPath('../data/Comp_CD3 APC-H7_F3_F03_022.fcs'),
 PosixPath('../data/Comp_CD45RA Ax700_E3_E03_021.fcs'),
 PosixPath('../data/Comp_HLA-DR BV786_C2_C02_011.fcs'),
 PosixPath('../data/Comp_Live Dead UV Blue_F4_F04_030.fcs'),
 PosixPath('../data/Comp_Unstained mouse_D4_D04_028.fcs'),
 PosixPath('../data/Comp_Unstained rat_E4_E04_029.fcs'),
 PosixPath('../data/PBMC_090120_1_DCs_A5_A05_031.fcs'),
 PosixPath('../data/PBMC_090120_2_DCs_B5_B05_032.fcs'),
 PosixPath('../data/PBMC_091520_1_DCs_C5_C05_033.fcs'),
 PosixPath('../data/PBMC_091520_2_DCs_D5_D05_034.fcs')]

## View a CSV from `data/`

Lists CSVs in `data/` and previews one. `pbmc_s1.csv` is ~2M rows, so only a
slice loads by default.

In [4]:
csv_files = sorted(DATA_DIR.glob("*.csv"))
print(csv_files)

CSV_PATH = csv_files[0]  # change the index (or set a path directly) to pick a different file
PREVIEW_ROWS = 5_000  # the full file can be ~2M rows — raise this or set to None to load it all

csv_df = pd.read_csv(CSV_PATH, nrows=PREVIEW_ROWS)
print(f"{CSV_PATH.name}: showing {len(csv_df):,} rows x {csv_df.shape[1]} columns")
csv_df.head()

[PosixPath('../data/pbmc_s1.csv')]
pbmc_s1.csv: showing 5,000 rows x 32 columns


,FSC-A,FSC-H,SSC-A,CADM1 FITC,CD141 BB630,CD123 BB660,FcER1a BB700,Unnamed: 7,XCR1 BV421,CD14 BV510,...,CD20 BUV805,CD11c APC,CD45RA Ax700,CD3 APC-H7,CD301 PE,Unnamed: 27,CD19 PE-Cy5,CD5 PE-Cy55,CD88 PE-Cy7,Time
0,87361.875,79434.130,598.27870,160.180850,59.948257,265.05234,2209.8892,618.39240,86.72689,135.51247,...,263.73828,134.774190,2193.2383,1619.1234,502.20874,186.362260,161.58138,3681.0247,3134.7780,0.036351
1,126908.380,108476.130,1030.25870,202.591660,1682.306000,2094.69120,4600.4336,1695.13060,1039.57170,1021.77970,...,1318.86440,1797.315000,3525.5068,677.8574,5200.44700,3191.416500,642.05490,2521.5664,2330.8674,0.036405
2,80773.440,72883.586,460.69922,63.809006,41.512314,187.33128,1758.5116,564.46765,85.91007,132.62085,...,163.51422,136.576170,2556.1055,1337.7683,291.15740,116.808624,78.40261,3396.2102,2594.3127,0.036667
3,98579.290,90491.320,798.33765,180.572540,584.399660,4503.69340,10570.9375,4350.17040,1614.76890,780.50134,...,487.67517,2861.196300,4038.2563,650.1272,547.43950,503.161380,774.61755,3809.5645,5784.4300,0.036894
4,81687.250,74994.590,586.49786,156.943650,-2.742810,89.63686,1083.4099,343.41837,75.98193,173.59355,...,174.48912,77.228905,3129.6992,1420.8434,196.80910,97.804460,122.22795,2326.3208,2200.1838,0.037151


## EDA: channel metadata

Classifies each CSV column:

- `FSC-A` / `FSC-H` / `SSC-A` — scatter, no target/marker
- `Time` — acquisition time, no target/marker
- `Unnamed: N` — blank header, no target/marker
- everything else — `"<target> <marker>"`, e.g. `CADM1 FITC` → target `CADM1`, marker `FITC`

In [5]:
SCATTER_TIME_COLUMNS = {"FSC-A": "scatter", "FSC-H": "scatter", "SSC-A": "scatter", "Time": "time"}
TARGET_MARKER_OVERRIDES = {
    "Live Dead UV Blue": ("Live Dead", "UV Blue"),  # viability dye, not a CD antigen
}


def classify_column(col):
    if col.startswith("Unnamed"):
        return pd.Series({"category": "empty", "target": None, "marker": None})
    if col in SCATTER_TIME_COLUMNS:
        return pd.Series({"category": SCATTER_TIME_COLUMNS[col], "target": None, "marker": None})
    target, marker = TARGET_MARKER_OVERRIDES.get(col, col.split(" ", 1))
    return pd.Series({"category": "target-marker", "target": target, "marker": marker})


metadata = pd.DataFrame({"column": csv_df.columns})
metadata = metadata.join(metadata["column"].apply(classify_column))
metadata

,column,category,target,marker
0,FSC-A,scatter,None,None
1,FSC-H,scatter,None,None
2,SSC-A,scatter,None,None
3,CADM1 FITC,target-marker,CADM1,FITC
4,CD141 BB630,target-marker,CD141,BB630
5,CD123 BB660,target-marker,CD123,BB660
6,FcER1a BB700,target-marker,FcER1a,BB700
7,Unnamed: 7,empty,None,None
8,XCR1 BV421,target-marker,XCR1,BV421
9,CD14 BV510,target-marker,CD14,BV510


## Demonstrating that the data is already compensated

`docs/DATASET.md` found an identity spillover matrix embedded in every
`.fcs` file — the file's own claim that no compensation is needed. This
section verifies that independently: compute a **real** matrix from the
single-stain `Comp_*.fcs` controls and apply it to the PBMC data. If the
data is already compensated, a second real correction should over-correct —
pushing values negative — rather than do nothing.

Controls are discovered by globbing `Comp_*.fcs` and reading each file's own
channel labels, so this scales to however many control files exist (locally,
9 of 28 fluorescence channels have one; the rest are left as identity and
reported).

In [6]:
import numpy as np
import plotly.graph_objects as go

# Reload the full PBMC sample (not the 5,000-row CSV preview) — docs/DATASET.md
# confirmed its raw values match pbmc_s1.csv exactly.
PBMC_PATH = [p for p in fcs_files if p.name.startswith("PBMC_")][0]
pbmc_sample = fk.Sample(PBMC_PATH)

# All non-scatter, non-time detectors — read from the file, not hardcoded.
FLUOR_DETECTORS = [c for c in pbmc_sample.pnn_labels if c not in ("FSC-A", "FSC-H", "SSC-A", "Time")]

# Detector -> human-readable name (e.g. "R670-A" -> "CD11c APC"), from $PnS.
FRIENDLY_NAME = {d: (m or d) for d, m in zip(pbmc_sample.pnn_labels, pbmc_sample.pns_labels)}

raw = pbmc_sample.as_dataframe(source="raw")
raw.columns = pbmc_sample.pnn_labels

print(f"PBMC file: {PBMC_PATH.name} ({len(raw):,} events, {len(FLUOR_DETECTORS)} fluorescence channels)")

PBMC file: PBMC_090120_1_DCs_A5_A05_031.fcs (2,088,689 events, 28 fluorescence channels)


In [7]:
# Each control's $PnS (marker name) is filled in only for its stained detector —
# use that to find the primary channel automatically, for any number of controls.
control_files = [p for p in fcs_files if p.name.startswith("Comp_") and "Unstained" not in p.name]

control_by_channel = {}
for path in control_files:
    control_sample = fk.Sample(path)
    primary_channel = next(
        (detector for detector, marker in zip(control_sample.pnn_labels, control_sample.pns_labels) if marker),
        None,
    )
    if primary_channel is None:
        print(f"skipping {path.name}: no labeled channel found")
        continue
    control_by_channel[primary_channel] = path

print(f"{len(control_by_channel)}/{len(FLUOR_DETECTORS)} channels have a local single-stain control:")
{FRIENDLY_NAME[ch]: p.name for ch, p in control_by_channel.items()}

9/28 channels have a local single-stain control:


{'CD11c APC': 'Comp_CD11c APC_D3_D03_020.fcs',
 'CD123 BB660': 'Comp_CD123 BB660_F2_F02_014.fcs',
 'CD14 BV510': 'Comp_CD14 BV510_G1_G01_007.fcs',
 'CD19 PE-Cy5': 'Comp_CD19 PE-Cy5_A3_A03_017.fcs',
 'CD20 BUV805': 'Comp_CD20 BUV805_E1_E01_005.fcs',
 'CD3 APC-H7': 'Comp_CD3 APC-H7_F3_F03_022.fcs',
 'CD45RA Ax700': 'Comp_CD45RA Ax700_E3_E03_021.fcs',
 'HLA-DR BV786': 'Comp_HLA-DR BV786_C2_C02_011.fcs',
 'Live Dead UV Blue': 'Comp_Live Dead UV Blue_F4_F04_030.fcs'}

In [8]:
def compute_spillover_row(control_path, primary_channel, detectors, pos_pct=90, neg_pct=10):
    """One spillover-matrix row: fraction of the primary channel's signal
    appearing in each other detector, from a single-stain control.

    Simplified (percentile gating, not dedicated gates) — illustrative, not a
    production-grade compensation matrix.
    """
    control_df = fk.Sample(control_path).as_dataframe(source="raw")
    control_df.columns = fk.Sample(control_path).pnn_labels

    primary_values = control_df[primary_channel]
    positive = primary_values > np.percentile(primary_values, pos_pct)
    negative = primary_values < np.percentile(primary_values, neg_pct)

    primary_baseline = control_df.loc[negative, primary_channel].median()
    primary_bright = control_df.loc[positive, primary_channel].median() - primary_baseline
    if primary_bright <= 0:
        raise ValueError(f"{control_path.name}: no separable positive population on {primary_channel}")

    row = {}
    for detector in detectors:
        baseline = control_df.loc[negative, detector].median()
        bright = control_df.loc[positive, detector].median() - baseline
        row[detector] = bright / primary_bright
    return pd.Series(row)


# Identity default — uncontrolled channels stay uncorrected rather than guessed.
spillover_matrix = pd.DataFrame(np.eye(len(FLUOR_DETECTORS)), index=FLUOR_DETECTORS, columns=FLUOR_DETECTORS)
uncontrolled_channels = []
for detector in FLUOR_DETECTORS:
    if detector in control_by_channel:
        spillover_matrix.loc[detector] = compute_spillover_row(control_by_channel[detector], detector, FLUOR_DETECTORS)
    else:
        uncontrolled_channels.append(detector)

print(f"{len(uncontrolled_channels)} channels left as identity (no local control): "
      f"{[FRIENDLY_NAME[d] for d in uncontrolled_channels]}")

19 channels left as identity (no local control): ['CADM1 FITC', 'CD141 BB630', 'FcER1a BB700', 'B780-A', 'XCR1 BV421', 'CD16 BV570', 'V605-A', 'CD172a BV650', 'V710-A', 'CD303 BV750', 'CD1c BUV395', 'U500-A', 'CD56 BUV563', 'CD89 BUV661', 'CD163 BUV737', 'CD301 PE', 'G610-A', 'CD5 PE-Cy55', 'CD88 PE-Cy7']


### Visualize the computed matrix

Heatmap of the 9 controlled channels — hover a cell for the exact spillover
fraction. Diagonal is always 1; brighter off-diagonal cells mean more
bleed-through between those two fluorophores.

In [9]:
controlled_channels = list(control_by_channel.keys())
controlled_matrix = spillover_matrix.loc[controlled_channels, controlled_channels]
controlled_labels = [FRIENDLY_NAME[d] for d in controlled_channels]

# Sequential single-hue ramp for magnitude — never a rainbow scale.
BLUE_SEQUENTIAL = [
    (0.00, "#cde2fb"), (0.15, "#9ec5f4"), (0.30, "#6da7ec"), (0.45, "#3987e5"),
    (0.60, "#2a78d6"), (0.75, "#1c5cab"), (0.90, "#104281"), (1.00, "#0d366b"),
]

heatmap_fig = go.Figure(go.Heatmap(
    z=controlled_matrix.values,
    x=controlled_labels,
    y=controlled_labels,
    colorscale=BLUE_SEQUENTIAL,
    zmin=0, zmax=1,
    hovertemplate="spillover from %{y}<br>into %{x}: %{z:.3f}<extra></extra>",
    colorbar=dict(title="fraction"),
))
heatmap_fig.update_layout(
    title="Computed spillover matrix (from single-stain controls)",
    xaxis_title="into channel", yaxis_title="from channel",
    template="simple_white", font_color="#0b0b0b",
    yaxis_autorange="reversed",
)
heatmap_fig.show()

### Apply it to the PBMC data and check for over-correction

`true_signal = observed @ inverse(spillover_matrix)`. Applied to data that's
already compensated, this double-corrects — spiking negative values.
`uncontrolled_channels` pass through unchanged, so the comparison below
covers only the 9 controlled channels.

In [10]:
M = spillover_matrix.loc[FLUOR_DETECTORS, FLUOR_DETECTORS].values
double_compensated = pd.DataFrame(raw[FLUOR_DETECTORS].values @ np.linalg.inv(M), columns=FLUOR_DETECTORS)

pct_negative = pd.DataFrame({
    "channel": controlled_labels,
    "already-compensated (raw/CSV)": [(raw[d] < 0).mean() * 100 for d in controlled_channels],
    "after applying computed matrix": [(double_compensated[d] < 0).mean() * 100 for d in controlled_channels],
})

overall_before = (raw[FLUOR_DETECTORS] < 0).values.mean() * 100
overall_after = (double_compensated[FLUOR_DETECTORS] < 0).values.mean() * 100
print(f"overall % negative across all 28 channels — before: {overall_before:.1f}%, after: {overall_after:.1f}%")
pct_negative.round(1)

overall % negative across all 28 channels — before: 2.4%, after: 22.8%


,channel,already-compensated (raw/CSV),after applying computed matrix
0,CD11c APC,1.9,32.5
1,CD123 BB660,1.9,50.4
2,CD14 BV510,1.0,33.9
3,CD19 PE-Cy5,4.1,6.4
4,CD20 BUV805,3.7,41.0
5,CD3 APC-H7,1.9,10.6
6,CD45RA Ax700,0.9,1.1
7,HLA-DR BV786,1.4,17.8
8,Live Dead UV Blue,0.9,4.0


In [11]:
compare_fig = go.Figure()
compare_fig.add_bar(
    name="already-compensated (raw/CSV)",
    x=pct_negative["channel"], y=pct_negative["already-compensated (raw/CSV)"],
    marker_color="#2a78d6",
    hovertemplate="%{x}<br>already-compensated: %{y:.1f}%<extra></extra>",
)
compare_fig.add_bar(
    name="after applying computed matrix",
    x=pct_negative["channel"], y=pct_negative["after applying computed matrix"],
    marker_color="#eb6834",
    hovertemplate="%{x}<br>after second compensation: %{y:.1f}%<extra></extra>",
)
compare_fig.update_layout(
    barmode="group", template="simple_white", font_color="#0b0b0b",
    title="Percent of events with a negative value, per channel",
    yaxis_title="% negative events", xaxis_title="channel",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
)
compare_fig.show()

### Conclusion

Applying an independently computed matrix pushes several channels from
~1-4% negative events to 30-50% — the double-compensation signature.
Combined with `docs/DATASET.md`'s identity-matrix finding, this confirms
`pbmc_s1.csv` was already compensated.

## Interactive: edit the matrix and explore the effect

Pick two markers, type a new spillover value, and click **Apply**. Both the
heatmap and the impact chart update live, across all 28 fluorescence
channels — not just the 9 with a real control — since a spillover edit on
one channel can shift the computed "true" value of any other channel.

Starts from the matrix computed above; **Reset matrix** restores it.

In [12]:
import ipywidgets as widgets
from IPython.display import display

ALL_LABELS = [FRIENDLY_NAME[d] for d in FLUOR_DETECTORS]
raw_values = raw[FLUOR_DETECTORS].values
baseline_pct_negative = (raw_values < 0).mean(axis=0) * 100

editable_matrix = spillover_matrix.copy()


def pct_negative_for_matrix(matrix_df):
    M = matrix_df.loc[FLUOR_DETECTORS, FLUOR_DETECTORS].values
    compensated = raw_values @ np.linalg.inv(M)  # raises LinAlgError if not invertible
    return (compensated < 0).mean(axis=0) * 100


def build_heatmap_fig(matrix_df):
    fig = go.Figure(go.Heatmap(
        z=matrix_df.values, x=ALL_LABELS, y=ALL_LABELS,
        colorscale=BLUE_SEQUENTIAL, zmin=0, zmax=1,
        hovertemplate="spillover from %{y}<br>into %{x}: %{z:.3f}<extra></extra>",
        colorbar=dict(title="fraction"),
    ))
    fig.update_layout(
        title="Editable spillover matrix (all 28 channels)",
        xaxis_title="into channel", yaxis_title="from channel",
        template="simple_white", font_color="#0b0b0b",
        yaxis_autorange="reversed", width=750, height=750,
    )
    return fig


def build_impact_fig(current_pct_negative):
    fig = go.Figure()
    fig.add_bar(name="already-compensated (raw)", y=ALL_LABELS, x=baseline_pct_negative,
                orientation="h", marker_color="#2a78d6")
    fig.add_bar(name="after applying matrix (current)", y=ALL_LABELS, x=current_pct_negative,
                orientation="h", marker_color="#eb6834")
    fig.update_layout(
        barmode="group", template="simple_white", font_color="#0b0b0b",
        title="Percent of events with a negative value, per channel",
        xaxis_title="% negative events", width=650, height=750,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
    )
    return fig


# Plain `go.Figure` + `Output`, not `go.FigureWidget` — FigureWidget needs `anywidget`
# on Plotly 7.x, which some notebook front ends (VS Code included) fail to render.
# Redrawing a plain figure inside an Output on every update sidesteps that entirely.
heatmap_output = widgets.Output()
impact_output = widgets.Output()


def render(matrix_df, current_pct_negative):
    heatmap_output.clear_output(wait=True)
    with heatmap_output:
        display(build_heatmap_fig(matrix_df))
    impact_output.clear_output(wait=True)
    with impact_output:
        display(build_impact_fig(current_pct_negative))


render(editable_matrix, pct_negative_for_matrix(editable_matrix))

In [13]:
from_dropdown = widgets.Dropdown(options=ALL_LABELS, description="From:")
to_dropdown = widgets.Dropdown(options=ALL_LABELS, description="Into:")
value_box = widgets.FloatText(value=0.0, description="New value:", step=0.01)
apply_button = widgets.Button(description="Apply", button_style="primary")
reset_button = widgets.Button(description="Reset matrix")
status = widgets.Output()


def on_apply(_):
    with status:
        status.clear_output()
        from_detector = FLUOR_DETECTORS[ALL_LABELS.index(from_dropdown.value)]
        to_detector = FLUOR_DETECTORS[ALL_LABELS.index(to_dropdown.value)]
        previous_value = editable_matrix.loc[from_detector, to_detector]
        editable_matrix.loc[from_detector, to_detector] = value_box.value
        try:
            updated_pct_negative = pct_negative_for_matrix(editable_matrix)
        except np.linalg.LinAlgError:
            editable_matrix.loc[from_detector, to_detector] = previous_value
            print("That edit made the matrix singular (not invertible) — change reverted.")
            return
        render(editable_matrix, updated_pct_negative)
        print(f"{from_dropdown.value} -> {to_dropdown.value} set to {value_box.value:.3f}")


def on_reset(_):
    with status:
        status.clear_output()
        editable_matrix.loc[:, :] = spillover_matrix.values
        render(editable_matrix, pct_negative_for_matrix(editable_matrix))
        print("Matrix reset to the computed values.")


apply_button.on_click(on_apply)
reset_button.on_click(on_reset)

controls = widgets.HBox([from_dropdown, to_dropdown, value_box, apply_button, reset_button])
widgets.VBox([controls, status, widgets.HBox([heatmap_output, impact_output])])